In [ ]:
import os
import json
from pathlib import Path
from agents.agentEvolver_v2.creator_agent import CreatorAgent
from langchain_core.messages import HumanMessage

# Instantiate the CreatorAgent
# This will create a new run directory for our evaluation
creator_agent = CreatorAgent()

# Get the run directory
run_dir = Path(creator_agent.run_dir)
print(f"Run directory: {run_dir}")

In [ ]:
# Create dummy files for evaluation
# The analyzer_node's tools read from these files in the run directory

# Create a dummy game run directory inside the main run_dir
game_run_id = "game_20250908_223025_fg"
game_run_dir = run_dir / game_run_id
game_run_dir.mkdir(exist_ok=True)

# 1. Dummy game_output.txt for the history
game_output_content = '''
GAME RESULTS:

results_file_path: /tmp/results_20250908_223025.json
Some game output...
Player FooPlayer had an error.
Defaulting to Random Action.
Choose action with score: 0
...'''
with open(game_run_dir / "game_output.txt", "w") as f:
    f.write(game_output_content)

# 2. Dummy foo_player.py for the history
foo_player_content = '''
from catanatron.models.enums import ActionType

class FooPlayer:
    def choose_action(self, game, playable_actions):
        # Some dummy logic from a previous evolution
        for action in playable_actions:
            if action.action_type == ActionType.BUILD_SETTLEMENT:
                return action
        return playable_actions[0]'''
with open(game_run_dir / "foo_player.py", "w") as f:
    f.write(foo_player_content)

# 3. Dummy results.json for the history
results_content = {"Player Summary": {"FooPlayer": {"WINS": 5, "AVG VP": 8.2}}}
with open(game_run_dir / "results.json", "w") as f:
    json.dump(results_content, f, indent=2)

# 4. Dummy performance_history.json
performance_history_content = {
    "Evolution 0": {
        "wins": 5,
        "avg_score": 8.2,
        "avg_turns": 50,
        "full_game_log_path": str(game_run_dir.relative_to(run_dir) / "game_output.txt"),
        "json_game_results_path": str(game_run_dir.relative_to(run_dir) / "results.json"),
        "cur_foo_player_path": str(game_run_dir.relative_to(run_dir) / "foo_player.py"),
        "timestamp": "2025-09-08 22:30:25"
    }
}
with open(run_dir / "performance_history.json", "w") as f:
    json.dump(performance_history_content, f, indent=2)

# 5. The "current" foo_player.py that the analyzer will look at.
current_foo_player_content = '''
from catanatron.models.enums import ActionType

class FooPlayer:
    def choose_action(self, game, playable_actions):
        # Some new, buggy logic
        print("This is a buggy player")
        return None # Oops, this will cause an error'''
# The read_foo() tool reads from FOO_TARGET_FILE in the agent's directory.
agent_dir = Path("agents/agentEvolver_v2")
with open(agent_dir / "foo_player.py", "w") as f:
    f.write(current_foo_player_content)

print("Dummy files created.")

In [ ]:
from langchain.evaluation import StringEvaluator
from agents.agentEvolver_v2.creator_agent import CreatorGraphState

# a. Prepare a mock CreatorGraphState
mock_state: CreatorGraphState = {
    "analyzer_messages": [],
    "recent_meta_message": HumanMessage(content='''
ANALYZER OBJECTIVE:

Analyze the most recent game run.
- If there are no syntax errors, provide the scores from the results file and a brief analysis of the game output.
- Highlight any errors, warnings, or signs of implementation issues in the `game_output.txt` file.
- If there is a syntax error, detail the error message, line number, and the problematic line of code.
- Keep the response concise.
'''),
    "meta_messages": [],
    "strategizer_messages": [],
    "researcher_messages": [],
    "coder_messages": [],
    "recent_helper_response": HumanMessage(content=""),
    "game_results": HumanMessage(content=""),
    "tool_calling_messages": [],
}

# b. Call the directly accessible analyzer_node method
updated_state = creator_agent.analyzer_node(mock_state)
analyzer_response = updated_state['recent_helper_response']

print("--- Analyzer Response ---")
print(analyzer_response.content)
print("------------------------")

# c. Use langchain.evaluation.StringEvaluator to check the analyzer's output
evaluator = StringEvaluator("cot_qa")

evaluation_result = evaluator.evaluate_strings(
    prediction=analyzer_response.content,
    input="Analyze the provided game logs and identify any errors or warnings.",
    reference="The analysis should mention that the player is 'Defaulting to Random Action' and 'Choose action with score: 0'."
)

# d. Print the evaluation results
print("--- Evaluation Result ---")
print(f"Score: {evaluation_result['score']}")
print(f"Reasoning: {evaluation_result['reasoning']}")
print("-------------------------")